# Goal of this notebook
- Brainstorm methods to preprocess fetched raw data in `data/01-raw` -- how do we handle duplicates? what format do we wanna store in? do we wanna store all relevant time features for our purposes?

- Given all successful raw runs, produce one T1 parquet dataset with schema `(location, valid_time, analysis_time, kindex, run_id).`

# Imports

In [10]:
import os
import json
import re

# object oriented approach to working with paths
from pathlib import Path

import duckdb
# import pyarrow
import pyarrow as pa
import pyarrow.parquet as pq

# Ensure current working directory is the root

In [2]:
os.chdir("..")

In [3]:
ls

 Volume in drive C is Windows-SSD
 Volume Serial Number is 42FE-2E01

 Directory of c:\Users\Keith\OneDrive\Post uni\Personal projects\space-weather-project

04/03/2026  15:23    <DIR>          .
20/01/2026  01:54    <DIR>          ..
04/03/2026  15:23                66 .gitignore
14/01/2026  15:05    <DIR>          .vscode
20/01/2026  02:09    <DIR>          code-diagrams
02/12/2025  20:29    <DIR>          config
06/01/2026  15:22    <DIR>          data
02/12/2025  18:44                 4 docker-compose.yml
02/12/2025  18:44                 4 Dockerfile
14/02/2026  03:52    <DIR>          entrypoint
02/12/2025  20:17    <DIR>          env
02/12/2025  18:30    <DIR>          models
10/02/2026  23:16    <DIR>          notebooks
20/01/2026  02:02               844 README.md
05/03/2026  15:24               365 requirements-dev.txt
02/12/2025  18:44                 4 requirements-prod.txt
05/03/2026  15:26    <DIR>          src
04/03/2026  15:23    <DIR>          tests
               6 Fi

In [6]:
Path.home()

WindowsPath('C:/Users/Keith')

# Familiarizing with `Pathlib` library

## Get cwd

In [61]:
Path.cwd()

WindowsPath('c:/Users/Keith/OneDrive/Post uni/Personal projects/space-weather-project')

### Make sure `Path(".").resolve() == Path.cwd()`

In [62]:
# p.resolve() to get absolute path from relative pathlib path object `p`

Path(".").resolve() == Path.cwd()

True

## Return string rep of a Path

In [63]:
Path.cwd().as_posix() 

'c:/Users/Keith/OneDrive/Post uni/Personal projects/space-weather-project'

## read_json MULTIPLE files [(see DuckDB doco here)](https://duckdb.org/docs/stable/data/json/loading_json#:~:text=Note%20that%20DuckDB%20can%20convert%20JSON%20arrays%20directly%20to%20its%20internal%20LIST%20type%2C%20and%20missing%20keys%20become%20NULL%3A)

## read_json by default AUTO infers schema [(see DuckDB doco here)](https://duckdb.org/docs/stable/data/json/loading_json#:~:text=BOOL-,true,-columns)

## COPY .. TO  exports data from DuckDB to external format [(see duckDB doco)](https://duckdb.org/docs/stable/sql/statements/copy#copy--to:~:text=COPY%20...%20TO-,COPY%20...%20TO,-exports%20data%20from)

# Write currently fetched datasets to parquet

In [49]:
FETCHED_K_INDEX_RELATIVE_DIR = Path(".") / "data/01-raw/space_weather/k_index"
# each subdirectory corresponds to a run
runs = list(FETCHED_K_INDEX_RELATIVE_DIR.iterdir())

# each run  contains fetched chunks, json manifest and _FAILURE or _SUCCESS status
list(
    runs[0].iterdir()
)


[WindowsPath('data/01-raw/space_weather/k_index/run_id=20260307T050056Z/chunk_20250101T000000Z__20250131T000000Z.jsonl'),
 WindowsPath('data/01-raw/space_weather/k_index/run_id=20260307T050056Z/chunk_20250131T000000Z__20250302T000000Z.jsonl'),
 WindowsPath('data/01-raw/space_weather/k_index/run_id=20260307T050056Z/chunk_20250302T000000Z__20250401T000000Z.jsonl'),
 WindowsPath('data/01-raw/space_weather/k_index/run_id=20260307T050056Z/_manifest.json'),
 WindowsPath('data/01-raw/space_weather/k_index/run_id=20260307T050056Z/_SUCCESS')]

In [ ]:
[x for x in (Path('.') / "notebooks" ).iterdir()]

[WindowsPath('notebooks/01_ingest_exploration.ipynb'),
 WindowsPath('notebooks/02_kindex_raw_to_preproc_exploration.ipynb')]

In [60]:
[p.as_posix() for p in FETCHED_K_INDEX_RELATIVE_DIR.rglob('*manifest.json')]

['data/01-raw/space_weather/k_index/run_id=20260307T050056Z/_manifest.json',
 'data/01-raw/space_weather/k_index/run_id=20260307T050221Z/_manifest.json',
 'data/01-raw/space_weather/k_index/run_id=20260307T050513Z/_manifest.json',
 'data/01-raw/space_weather/k_index/run_id=20260307T051543Z/_manifest.json']

## Glob all jsonl chunks, try to save to notebook folder first

In [ ]:
# get all .jsonl chunks across all runs (rglob used since chunks are located INSIDE a run directory)
# as_posix() returns string representation 

sources = [p.as_posix() for p in FETCHED_K_INDEX_RELATIVE_DIR.rglob('*.jsonl')]
output_path = (Path('.') / "notebooks" / "test_kindex_T1_no_loc.parquet").as_posix() 

# Execute the conversion
# union_by_name=True ensures it handles schema mismatches across files
duckdb.execute(f"""
    COPY (
        SELECT regexp_extract(filename, '.+/run_id=(.+)/.+', 1) AS run_ID, index, valid_time, analysis_time
        FROM read_json_auto({sources}, union_by_name=True)
    ) TO '{output_path}' (FORMAT PARQUET);
""")

duckdb.read_parquet(output_path).to_df()

,run_ID,index,valid_time,analysis_time
0,20260307T050056Z,3,2025-01-01 00:00:00,2025-01-02 03:53:49
1,20260307T050056Z,5,2025-01-01 03:00:00,2025-01-02 03:53:49
2,20260307T050056Z,5,2025-01-01 06:00:00,2025-01-02 03:53:49
3,20260307T050056Z,5,2025-01-01 09:00:00,2025-01-02 03:53:49
4,20260307T050056Z,6,2025-01-01 12:00:00,2025-01-02 03:53:49
...,...,...,...,...
3652,20260307T050513Z,4,2025-12-31 12:00:00,2026-01-01 00:17:13
3653,20260307T050513Z,2,2025-12-31 15:00:00,2026-01-01 00:17:13
3654,20260307T050513Z,1,2025-12-31 18:00:00,2026-01-01 00:17:13
3655,20260307T050513Z,1,2025-12-31 21:00:00,2026-01-01 00:17:13


## Include location from _manifest.json metadata via joins

First try to parse all manifest json files to extract location

In [106]:
# use the columns param in read_json* to ignore other keys in the _manifest.json files. 

query = f"""
SELECT created_at_utc, location, status
FROM read_json_auto({[p.as_posix() for p in FETCHED_K_INDEX_RELATIVE_DIR.rglob('*manifest.json')]},
    columns = {{created_at_utc: 'STRING', location: 'STRING', status: 'STRING'}}, 
    union_by_name=True, auto_detect=False)
"""
duckdb.execute(query).fetch_df()

,created_at_utc,location,status
0,20260307T050056Z,Australian region,SUCCESS
1,20260307T050221Z,Australian region,SUCCESS
2,20260307T050513Z,Australian region,SUCCESS
3,20260307T051543Z,Australian region,SUCCESS
4,20260312T003937Z,Australian regionnnn,FAILED
5,20260312T004356Z,Melbourne,SUCCESS
6,20260312T005243Z,Alice Springs,SUCCESS


In [115]:
import duckdb

FETCHED_K_INDEX_RELATIVE_DIR = Path(".") / "data/01-raw/space_weather/k_index"

# note that empty data -> empty jsonl -> meaning it will simply be omitted -- do we WANT this behavior though? 
sources = [p.as_posix() for p in FETCHED_K_INDEX_RELATIVE_DIR.rglob('*.jsonl')]

# include success indicator!!!

query_true_T1 = f"""
WITH
    locs AS (
        SELECT created_at_utc, location, status
        FROM read_json_auto({[p.as_posix() for p in FETCHED_K_INDEX_RELATIVE_DIR.rglob('*manifest.json')]},
        columns = {{created_at_utc: 'STRING', location: 'STRING', status: 'STRING'}}, union_by_name=True, auto_detect=False)
        WHERE status = 'SUCCESS'
    ),
    T1_no_loc AS (
        SELECT regexp_extract(filename, '.+/run_id=(.+)/.+', 1) AS run_ID, index, valid_time, analysis_time
        FROM read_json_auto({sources}, union_by_name=True)
    )

SELECT run_ID, valid_time, analysis_time, location, index, status
FROM T1_no_loc LEFT JOIN locs
ON T1_no_loc.run_ID = locs.created_at_utc;
"""

T1_test = duckdb.execute(query_true_T1).fetch_df()
#print(T1_test)
display(T1_test)
display(T1_test[['run_ID', 'location']].drop_duplicates().rename(columns={'location':f'location ({T1_test.location.nunique()} locs)'}))

,run_ID,valid_time,analysis_time,location,index,status
0,20260307T050221Z,2025-03-31 00:00:00,2025-04-01 00:18:20,Australian region,1,SUCCESS
1,20260307T050221Z,2025-03-31 03:00:00,2025-04-01 00:18:20,Australian region,2,SUCCESS
2,20260307T050221Z,2025-03-31 06:00:00,2025-04-01 00:18:20,Australian region,2,SUCCESS
3,20260307T050221Z,2025-03-31 09:00:00,2025-04-01 00:18:20,Australian region,2,SUCCESS
4,20260307T050221Z,2025-03-31 12:00:00,2025-04-01 00:18:20,Australian region,1,SUCCESS
...,...,...,...,...,...,...
11668,20260312T010212Z,2025-11-30 09:00:00,2025-12-01 00:05:29,Darwin,2,SUCCESS
11669,20260312T010212Z,2025-11-30 12:00:00,2025-12-01 00:05:29,Darwin,1,SUCCESS
11670,20260312T010212Z,2025-11-30 15:00:00,2025-12-01 00:05:29,Darwin,2,SUCCESS
11671,20260312T010212Z,2025-11-30 18:00:00,2025-12-01 00:05:29,Darwin,2,SUCCESS


,run_ID,location (4 locs)
0,20260307T050221Z,Australian region
480,20260307T050056Z,Australian region
2160,20260307T050513Z,Australian region
3360,20260312T005243Z,Alice Springs
4320,20260312T005945Z,Cocos Island
7200,20260312T010212Z,Darwin
9872,20260307T051543Z,Australian region
